In [ ]:

import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell
from Training.MainTrain import NN_Trainer, TrainingConfig
from Decoder_CLasses import ContinuousVoronoiDecoder, Phase1VoronoiDecoder
from Decoder_CLasses.NearUniformHoneycombBaseline import NearUniformHoneycombBaseline
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import IntrinsicSurfaceHoneycombBaseline
from Pred_NN_Classes.ppnet import PPNet
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d, cKDTree,QhullError
from Utils.ExportAbaqus_VoxelBased import export_abaqus_voxel_fem
from Utils.ExportAbaqus import export_abaqus_shell

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)

%matplotlib inline
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

In [ ]:
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
ConeTaped = TesPartsDir / "ConeTaped.stp"
FreeFormCLosed = TesPartsDir / "FreeFormClosed.stp"
Planar = TesPartsDir / "Planar.stp"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"
FreeFormSurf4 = TesPartsDir / "FreeForm4.stp"
FreeFormBench = TesPartsDir / "FreeFormBench.stp"
FreeSharp1 = TesPartsDir / "FreeSharp1.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
SeatBr = TesPartsDir / "SeatBr.stp"
MouseBot = TesPartsDir / "MouseBot.stp"
CircularHoles05 = TesPartsDir / "CircularHoles05.stp"
Helmet = TesPartsDir / "Helmet_New.stp"
FreeForm_Bend = TesPartsDir / "FreeForm_bend.stp"
CircBracket_Half = TesPartsDir / "CircBracket_Half.stp"
HelmetNew_Half = TesPartsDir / "HelmetNew_Half.stp"
Wing_NACA2412 = TesPartsDir / "Wing_NACA2412.stp"

shape_path = Planar

Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale = 0.5,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(shape_path)

dx,dy,dz = Face_Cad.print_face_info();



In [ ]:
voxel_size = float((dx + dy + dz) / 120.0)
thickness = 3.0 * voxel_size
print(f"dx:{dx} ,dy:{dy} ,dz:{dz}")
print(f"voxel size : {voxel_size}")
print(f"thickness : {thickness}")

In [ ]:

common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    faces_index_base=0,
)

shell_problem = ThickenShell(
    **common_settings
)

shell_problem.show_boundaries(
    segments_per_curve=2
)

In [ ]:
Boundary_loads = [

    {
        "type": "force",
        "boundaries": [5],
        "magnitude": 1,
        "direction": "x",
    },
    #{
    #     "type": "distributed_force",
    #     "boundaries": [0,1],
    #     "magnitude": 5.0,
    #     "direction": "y",
    #     "distribution": "parabolic",
    # },
]
LoadingCase = (f"Tensile(1N)-Voxel size:{voxel_size:.2f}mm-Thickness:{thickness:.2f}mm")
print(f"LoadingCase: {LoadingCase}")

BC_Report = shell_problem.boundary_loading(
    fixed=[0,1],
    loads=Boundary_loads,
    band=0.05* voxel_size,
)
loading_img = shell_problem.show_voxels_surface_and_bc(
    return_img=True,
    off_screen=True,
    window_size=(560, 430),
    show=True,
)

In [ ]:
# ============================================================
# CASE
# ============================================================

Seed_number = 40
strut_thickness = 0.40

Case_name = shape_path.stem

Explanation = (
    f"_{Seed_number}_Sth{str(strut_thickness)[2:]}_"
    f"{LoadingCase}_NEW"
)

timelapse_output_folder = f"Case_Studies/{Case_name}{Explanation}"
viz = CADVisualizer()

# ============================================================
# FEM MODEL
# ============================================================

fem = run_fem_loss.NeuralTOMOFEM(
    shell_problem,
    device=device,
    isotropic=False,
)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

cfg = TrainingConfig(

    # ========================================================
    # CASE INFORMATION
    # ========================================================

    LoadingCase=LoadingCase,


    # ========================================================
    # OPTIMIZATION MODE
    # ========================================================
    #
    # target_length_constrained_displacement
    #
    # BOTH PHASES:
    #   minimize displacement
    #   + CVT regularization
    #   + HybridME / FEM terms
    #   subject to:
    #       target total-length band
    #       stress constraint
    #       displacement constraint
    #
    # Phase 1:
    #   implicit continuous Voronoi representation
    #   continuous physical total length
    #   no explicit seed-spacing loss
    #   min seed distance excluded from feasibility
    #
    # Phase 2:
    #   explicit Voronoi graph
    #   exact physical total graph length
    #   seed separation active
    #   min seed distance included in feasibility
    #   explicit cell-edge regularization active
    #
    # ========================================================

    optimization_mode="minimize_displacement_at_target_length",

    displacement_objective_weight=1.0,

    # Smooth differentiable displacement objective.
    displacement_objective_mode="p_norm",


    # ========================================================
    # SHARED OBJECTIVE WEIGHTS
    # ========================================================
    #
    # These are now COMMON to Phase 1 and Phase 2.
    # Do not use old phase1_cvt_weight / phase2_cvt_weight,
    # phase1_enable_fem / phase2_enable_fem, etc.
    #
    # ========================================================

    fem_enabled=True,

    # Weak geometric regularization in both phases.
    cvt_weight=0.05,

    # In target-length mode this is the coefficient applied to
    # the common target-length-band penalty.
    #
    # Phase 1 length:
    #   continuous_total_curve_length
    #
    # Phase 2 length:
    #   exact total graph/network length
    #
    total_fiber_length_weight=1.0,


    # ========================================================
    # TARGET TOTAL-LENGTH CONSTRAINT
    # ========================================================
    #
    # Definitions:
    #
    #   lower = target_total_length
    #           - target_length_lower_tolerance
    #
    #   upper = target_total_length
    #           - target_length_upper_buffer
    #
    # Here:
    #
    #   lower = 120 - 15.0 = 105.0
    #   upper = 120 - 5.1  = 114.9
    #
    # Accepted:
    #
    #       135.0 <= L_total <= 144.9
    #
    # Same band is used in BOTH phases.
    #
    # ========================================================

    target_total_length=120.0,

    target_length_lower_tolerance=15.0,
    target_length_upper_buffer=5.1,

    target_length_under_weight=1.0,
    target_length_over_weight=1.0,


    # ========================================================
    # PHASE 1 — IMPLICIT CONTINUOUS DECODER
    # ========================================================
    #
    # These are REPRESENTATION parameters, not objective weights.
    #
    # ========================================================

    phase1_point_chunk_size=256,

    # Soft Voronoi assignment temperature.
    phase1_tau=0.01,

    # Physical implicit tube transition width.
    phase1_tube_beta=0.01,

    # Physical duplicate threshold:
    #
    #   duplicate distance
    #     = factor * strut thickness
    #
    #   = 1.10 * 0.40
    #   = 0.44 physical units
    #
    # The corrected decoder evaluates this in physical space,
    # not normalized UV.
    phase1_duplicate_merge_distance_factor=1.10,

    # Hard handoff criteria used only when entering Phase 2.
    phase1_handoff_activity_threshold=0.05,
    phase1_handoff_territory_ratio=0.20,

    # Keep Phase-1 seeds within the CAD domain.
    stage1_allow_seed_outside_domain=False,


    # ========================================================
    # PHASE 2 — EXPLICIT GRAPH CONTROLS
    # ========================================================

    # Seed-spacing term is intentionally zero in Phase 1
    # internally and active here in Phase 2.
    phase2_seed_spacing_weight=1.0,

    # PHYSICAL minimum seed spacing for Phase-2 feasibility.
    #
    # strut thickness      = 0.40
    # duplicate threshold = 0.44
    # required P2 spacing  = 0.60
    #
    phase2_min_seed_spacing=0.60,

    # Explicit graph-only cell-edge uniformity regularization.
    phase2_cell_edge_uniformity_weight=0,

    # Keep Phase-2 seeds trainable.
    stage2_freeze_seeds=False,

    stage2_allow_seed_outside_domain=False,


    # ========================================================
    # PHASE DURATIONS
    # ========================================================

    # ---------------- Phase 1 ----------------

    stage1_min_steps=100,
    stage1_max_steps=300,
    stage1_patience=150,

    stage1_min_delta_abs=1e-4,
    stage1_min_delta_rel=1e-3,

    stage1_scheduler_milestones=(0.60, 0.85),


    # ---------------- Phase 2 ----------------

    stage2_min_steps=100,
    stage2_max_steps=300,
    stage2_patience=50,

    stage2_min_delta_abs=1e-4,
    stage2_min_delta_rel=1e-3,

    stage2_scheduler_milestones=(0.70, 0.90),


    # ========================================================
    # PHASE TRANSITION
    # ========================================================

    restore_transition_checkpoint=True,

    # Rebuild optimizer because Phase 2 uses a different
    # representation and potentially a different seed count.
    reset_optimizer_between_stages=True,

    # Leave disabled unless explicit topology changes require
    # optimizer reinitialization during Phase 2.
    reset_optimizer_on_topology_change=False,

    # Pick the Phase-1 checkpoint according to the objective
    # used by the following phase.
    stage_transition_selection="next_stage_objective",

    stage_topology_grace_steps=20,
    stage_topology_grace_max_resets=5,

    debug_stage_controller=False,


    # ========================================================
    # INITIALIZATION / WARM START
    # ========================================================

    warm_start_optimized_function_path=None,

    warm_start_prune_inactive_seeds=False,

    # Fresh optimization should execute Phase 1.
    warm_start_skip_stage1=False,


    # ========================================================
    # SEED INITIALIZATION
    # ========================================================

    seed_number=Seed_number,

    use_balanced_seed_init=True,

    seed_init_fps_seed=11,

    strut_thickness=strut_thickness,


    # ========================================================
    # EXPLICIT GRAPH / EDGE SETTINGS
    # ========================================================
    #
    # Primarily used by Phase 2.
    #
    # ========================================================

    Edge_in_losses="all",

    decoder_use_trim_activity=True,

    generate_decoder_density_fiber=True,


    # ========================================================
    # PHASE-2 SEED-SPACING PENALTY DETAILS
    # ========================================================

    seed_spacing_power=2.0,

    seed_spacing_safety_factor=1.05,

    seed_spacing_aggregate_temperature=0.005,

    # Smooth preventive repulsion before the hard limit.
    seed_repulsion_factor=1.25,

    seed_repulsion_temperature_ratio=0.05,

    seed_repulsion_weight=0.10,


    # ========================================================
    # CVT DETAILS
    # ========================================================

    cvt_temperature=0.01,

    # Spatially emphasize CVT according to FEM displacement.
    fem_cvt_displacement_importance=True,

    fem_cvt_displacement_importance_power=1.0,

    fem_cvt_displacement_importance_floor=0.05,


    # ========================================================
    # EXPLICIT CELL REGULARIZATION DETAILS
    # ========================================================
    #
    # These matter when:
    #
    #   phase2_cell_edge_uniformity_weight > 0
    #
    # Phase 1 has no explicit cell-edge graph and therefore
    # these do not contribute there.
    #
    # ========================================================

    lam_cell_angle_uniform=0.5,

    lam_cell_radial_uniform=0.5,

    curve_length_tolerance=0.10,

    curve_length_outlier_weight=5.0,


    # ========================================================
    # DENSITY FILTER / PROJECTION
    # ========================================================
    #
    # Both decoders feed the SAME external FEM density
    # post-processing.
    #
    # The internal Phase-1 projection remains disabled.
    #
    # ========================================================

    use_3d_density_filter=False,

    filter_projection_strength=1.0,

    filter_projection_beta=8.0,

    filter_projection_eta=0.5,

    visualize_filtered_density=True,

    visualize_raw_density=False,


    # ========================================================
    # FEM PHYSICAL LIMITS
    # ========================================================

    fem_max_displacement=20.0,

    fem_yield_strength=45000.0,

    # Preventive mechanical penalty starts at 90% of hard limit.
    fem_training_safety_factor=0.90,

    # Stress is evaluated only where material density is
    # sufficiently meaningful.
    fem_stress_density_threshold=0.5,


    # ========================================================
    # FEM MATERIAL INTERPOLATION
    # ========================================================

    fem_penal=3.0,

    fem_rho_min_ratio=1e-4,

    fem_rho_min_start=1e-3,

    fem_rho_min_end=1e-4,

    fem_density_floor=0.0,


    # ========================================================
    # HYBRIDME / FEM LOSS COEFFICIENTS
    # ========================================================

    # Hard mechanical constraint coefficient.
    fem_constraint_weight=2.0,

    # Small baseline structural-improvement term.
    fem_baseline_weight=0.001,

    # Preventive safety-margin term.
    fem_safety_margin_weight=0.05,

    # Smooth approximation used for mechanical constraint.
    fem_constraint_p_norm=12.0,

    # Hard violation shaping.
    fem_violation_power=3.0,

    # Individual mechanical terms.
    stress_constraint_weight=2.0,

    displacement_constraint_weight=2.0,

    stress_reduction_weight=0.02,

    displacement_reduction_weight=0.05,

    stress_margin_weight=0.05,

    displacement_margin_weight=0.05,


    # ========================================================
    # AUTOMATIC CONSTRAINT CONTROLLER
    # ========================================================

    constraint_control="automatic",

    automatic_penalty_min=1e-3,

    automatic_penalty_max=1e6,

    automatic_penalty_growth=1.10,

    automatic_penalty_decay=0.98,

    automatic_penalty_gradient_ratio=2.0,

    automatic_secondary_gradient_fraction=0.25,

    automatic_secondary_weight_min=0.0,

    automatic_secondary_weight_max=0.25,


    # ========================================================
    # LEGACY ADAPTIVE FEM-LAMBDA
    # ========================================================
    #
    # Disabled because automatic constraint control is active.
    #
    # ========================================================

    adaptive_fem_penalty=False,

    fem_lambda_initial=1.0,

    fem_lambda_min=1.0,

    fem_lambda_max=1e3,

    fem_lambda_growth=1.02,

    fem_lambda_decay=0.99,

    fem_constraint_tolerance=0.0,


    # ========================================================
    # TARGET-LENGTH DOMAIN LOCK / PRUNING
    # ========================================================
    #
    # Leave disabled during initial validation of the new
    # two-stage workflow.
    #
    # ========================================================

    lock_seed_domain_after_target_length_feasible=False,

    target_length_lock_patience=25,

    target_length_unlock_on_violation=False,

    prune_outside_seeds_after_target_length_feasible=False,

    min_seed_count_after_target_length_prune=3,


    # ========================================================
    # INVALID FEM HANDLING
    # ========================================================

    skip_bad_fem_steps=True,

    invalid_fem_patience=3,

    invalid_lr_factor=0.5,

    minimum_learning_rate=1e-7,

    invalid_fem_consumes_budget=True,

    debug_fem_integrity=False,

    save_fem_debug_history=False,


    # ========================================================
    # SEED MOTION / PPNET
    # ========================================================

    use_independent_seed_offsets=True,

    independent_seed_offset_max=0.10,

    seed_offset_scale_start=0.10,

    seed_offset_scale_final=0.05,

    seed_offset_scale_ramp_frac=0.80,

    # Overall capability. Stage-specific settings above still
    # prohibit leaving the domain in this run.
    allow_seed_outside_domain=True,

    allow_seed_outside_domain_warmup_frac=0.001,


    # ========================================================
    # ROLLING SEED ANCHORS
    # ========================================================

    use_rolling_seed_anchors=True,

    seed_anchor_warmup_frac=0.0,

    seed_anchor_momentum=0.05,

    anchor_guard_updates=False,

    disable_rolling_seed_anchors_for_curve_only=False,


    # ========================================================
    # LEARNING RATES
    # ========================================================

    lr_seed_refine=5e-4,

    lr_independent_seed_offsets=5e-3,

    lr_delta_head=5e-4,

    lr_mlp=5e-4,

    lr_decoder=5e-4,


    # ========================================================
    # GENERAL NUMERICAL SETTINGS
    # ========================================================

    eps=1e-12,

    normalize_losses=True,

    grad_clip_norm=1.0,

    scheduler_gamma=0.5,


    # ========================================================
    # LOGGING / VISUALIZATION
    # ========================================================

    log_every=25,

    tensorboard_enabled=True,

    tensorboard_log_root="runs",

    experiment_name=f"{Case_name}{Explanation}",

    MakeTimelaps=True,

    timelapse_frame_step=50,

    timelapse_output_folder=timelapse_output_folder,
)


# ============================================================
# TRAINER
# ============================================================

trainer = NN_Trainer(
    generator=Face_Cad,
    viz=viz,

    # This remains the Phase-2 decoder class.
    # MainTrain constructs Phase1VoronoiDecoder internally.
    decoder_cls=ContinuousVoronoiDecoder,

    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
    face_mesh=face_mesh,
)


# ============================================================
# TRAIN
# ============================================================

result = trainer.train(
    shape_path=shape_path,
    face_tensors=[face_mesh],
)


# ============================================================
# SUMMARY
# ============================================================

print("best step:", result["best_step"])
print("best score:", result["best_score"])

print("density:", result["Final_shape_density"].shape)
print("fiber:", result["Final_shape_fiber_direction"].shape)

print("seed UV:", result["best_seed_points_uv"].shape)
print("seed XYZ:", result["best_seed_points_xyz"].shape)